# Rustren Studio — Offline Asset Catalog Baker (Colab GPU)

Pre-bakes **18 lightweight, brand-customizable WebP assets** + `catalog.json` for [Rustren Studio](https://nvm.web.id/rustren/studio/):
- **8 Neutral 3D / Glass / Studio Backdrops** (tintable in Studio via `blend_mode`)
- **6 Monochrome Optical Overlays** (flare, bokeh, 35mm grain, dust, CRT scanlines, vignette)
- **4 High-Contrast Luma Mattes** (ink bleed, brush sweep, smoke, geometric split — ready for `track_matte: "luma"`)

**Recommended runtime:** L4 / A100 / T4 GPU (~1–3 minutes total, no HuggingFace token required).

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors pillow numpy
import os, json, zipfile, torch
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from diffusers import AutoPipelineForText2Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'Using device: {device} ({torch.cuda.get_device_name(0) if device == "cuda" else "CPU"})')

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=dtype,
    variant='fp16' if device == 'cuda' else None,
).to(device)
pipe.set_progress_bar_config(disable=True)
print('SDXL-Turbo loaded!')

In [ ]:
OUT_DIR = 'rustren_catalog'
MEDIA_DIR = os.path.join(OUT_DIR, 'media')
THUMB_DIR = os.path.join(OUT_DIR, 'thumbs')
os.makedirs(MEDIA_DIR, exist_ok=True)
os.makedirs(THUMB_DIR, exist_ok=True)

catalog = []

def save_asset(asset_id, kind, title, tags, img, blend='normal', tintable=True, track_matte=None):
    img_full = img.convert('RGB').resize((1280, 720), Image.Resampling.LANCZOS)
    img_thumb = img_full.resize((320, 180), Image.Resampling.LANCZOS)
    media_rel = f'media/{asset_id}.webp'
    thumb_rel = f'thumbs/{asset_id}.webp'
    img_full.save(os.path.join(OUT_DIR, media_rel), 'WEBP', quality=82, method=6)
    img_thumb.save(os.path.join(OUT_DIR, thumb_rel), 'WEBP', quality=75, method=6)
    entry = {
        'id': asset_id,
        'kind': kind,
        'title': title,
        'tags': tags,
        'thumb': thumb_rel,
        'src': media_rel,
        'blend': blend,
        'tintable': tintable,
    }
    if track_matte:
        entry['track_matte'] = track_matte
    catalog.append(entry)
    print(f'  [OK] {asset_id} ({kind})')

def gen_sdxl(prompt, seed, monochrome=False):
    gen = torch.Generator(device=device).manual_seed(seed)
    # SDXL-Turbo works best at 512x512 or 768x448 then upscaled cleanly
    img = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0, width=768, height=448, generator=gen).images[0]
    if monochrome:
        img = img.convert('L').convert('RGB')
    return img

# 1. 8 Neutral 3D / Glass / Studio Backdrops
BACKDROPS = [
    ('bg-dark-glass', 'Dark Glass Monoliths', ['dark', '3d', 'glass'], 101, False,
     'minimalist dark architectural frosted glass monoliths, subtle rim lighting, monochrome dark charcoal background, octane render, studio lighting, no text, clean negative space'),
    ('bg-clay-stage', '3D Clay Cyclorama', ['neutral', '3d', 'tintable'], 102, True,
     'minimalist neutral matte white 3d studio cyclorama stage with soft global illumination shadows, subtle geometric pedestal, architectural photography, no text, clean'),
    ('bg-brushed-metal', 'Brushed Titanium', ['dark', 'metal', 'texture'], 103, True,
     'macro photography of dark brushed titanium anodized aluminum metal surface with subtle diagonal studio light reflection, minimalist industrial texture, no text'),
    ('bg-liquid-chrome', 'Liquid Chrome Waves', ['3d', 'chrome', 'abstract'], 104, True,
     'abstract minimalist monochrome liquid chrome mercury waves on pure black background, smooth specular highlights, luxury 3d render, no text'),
    ('bg-editorial-paper', 'Editorial Paper', ['light', 'paper', 'minimal'], 105, False,
     'minimalist warm off-white fine art cotton paper texture with very subtle soft directional window shadow, editorial design backdrop, no text'),
    ('bg-frosted-prism', 'Frosted Prism Light', ['glass', 'light', 'abstract'], 106, True,
     'abstract frosted ribbed glass refraction with soft monochrome light beams on dark background, minimalist cinema backdrop, no text'),
    ('bg-obsidian-waves', 'Obsidian Sculptural', ['dark', '3d', 'minimal'], 107, True,
     'minimalist dark matte obsidian sculptural curved ribbons, dramatic soft top lighting, luxury tech keynote background, no text'),
    ('bg-concrete-light', 'Brutalist Concrete', ['architecture', 'neutral', 'texture'], 108, True,
     'minimalist smooth architectural concrete wall with diagonal sunlit shadow line, brutalist editorial background, monochrome neutral gray, no text'),
]

print('Baking 8 Backdrops...')
for aid, title, tags, seed, mono, prompt in BACKDROPS:
    img = gen_sdxl(prompt, seed, monochrome=mono)
    save_asset(aid, 'backdrop', title, tags, img, blend='normal', tintable=mono)

# 2. 6 Monochrome Optical Overlays (for screen / overlay / multiply)
print('Baking 6 Optical Overlays...')
# GPU-baked optical flares & bokeh
OPTICAL_GPU = [
    ('ov-anamorphic-flare', 'Anamorphic Lens Flare', ['flare', 'optical', 'screen'], 201, 'screen',
     'pure white horizontal anamorphic cinema lens flare streak across pure pitch black background, high contrast monochrome optical flare, no objects, no text'),
    ('ov-bokeh-soft', 'Soft Optical Bokeh', ['bokeh', 'light', 'screen'], 202, 'screen',
     'soft out of focus white circular bokeh light orbs floating on pure pitch black background, monochrome cinema overlay, no text'),
    ('ov-dust-scratches', 'Analog Film Dust', ['vintage', 'dust', 'screen'], 203, 'screen',
     'subtle fine white analog 35mm film dust particles and tiny light specks on pure pitch black background, high contrast monochrome texture, no text'),
]
for aid, title, tags, seed, blend, prompt in OPTICAL_GPU:
    img = gen_sdxl(prompt, seed, monochrome=True)
    save_asset(aid, 'overlay', title, tags, img, blend=blend, tintable=True)

# Precision procedural overlays (35mm grain, CRT scanlines, soft vignette)
rng = np.random.default_rng(42)
grain = np.clip(128 + rng.normal(0, 26, (720, 1280)), 0, 255).astype(np.uint8)
save_asset('ov-film-grain', 'overlay', '35mm Cinema Grain', ['grain', 'film', 'overlay'],
           Image.fromarray(grain, 'L').convert('RGB'), blend='overlay', tintable=False)

crt = np.full((720, 1280), 180, dtype=np.uint8)
crt[::3, :] = 65
save_asset('ov-crt-scanlines', 'overlay', 'CRT Scanlines', ['crt', 'retro', 'overlay'],
           Image.fromarray(crt, 'L').convert('RGB'), blend='overlay', tintable=False)

yy, xx = np.mgrid[0:720, 0:1280]
dist = np.sqrt(((xx - 640) / 640.0) ** 2 + ((yy - 360) / 360.0) ** 2)
vig = np.clip(255 * (1.0 - 0.55 * np.clip(dist - 0.45, 0, 1.0) ** 1.6), 40, 255).astype(np.uint8)
save_asset('ov-vignette-soft', 'overlay', 'Cinematic Vignette', ['vignette', 'cinema', 'multiply'],
           Image.fromarray(vig, 'L').convert('RGB'), blend='multiply', tintable=False)

# 3. 4 High-Contrast Luma Mattes (for track_matte: 'luma')
print('Baking 4 Luma Mattes...')
MATTES = [
    ('mt-ink-bleed', 'Ink Bleed Mask', ['ink', 'organic', 'matte'], 301,
     'pure white organic ink blot spreading from center on pure pitch black background, stark high contrast black and white silhouette mask, no gray midtones, no text'),
    ('mt-brush-sweep', 'Dry Brush Sweep', ['brush', 'editorial', 'matte'], 302,
     'bold horizontal pure white dry paint brush stroke across center on pure pitch black background, high contrast black and white mask, no text'),
    ('mt-smoke-reveal', 'Smoke Cloud Mask', ['smoke', 'soft', 'matte'], 303,
     'soft luminous pure white smoke cloud billowing in center on pure pitch black background, high contrast monochrome luma matte, no text'),
]
for aid, title, tags, seed, prompt in MATTES:
    img = gen_sdxl(prompt, seed, monochrome=True)
    save_asset(aid, 'matte', title, tags, img, blend='normal', tintable=False, track_matte='luma')

# Procedural crisp diagonal split matte
diag = Image.new('L', (1280, 720), 0)
draw = ImageDraw.Draw(diag)
draw.polygon([(180, 0), (1100, 0), (920, 720), (0, 720)], fill=255)
diag = diag.filter(ImageFilter.GaussianBlur(radius=4))
save_asset('mt-geometric-split', 'matte', 'Diagonal Split Mask', ['geometric', 'clean', 'matte'],
           diag.convert('RGB'), blend='normal', tintable=False, track_matte='luma')

# Write catalog.json
with open(os.path.join(OUT_DIR, 'catalog.json'), 'w') as f:
    json.dump(catalog, f, indent=2)

# Zip everything
zip_path = 'rustren_catalog.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for fn in sorted(files):
            full = os.path.join(root, fn)
            zf.write(full, os.path.relpath(full, OUT_DIR))

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'\nDONE! {len(catalog)} assets baked into {zip_path} ({size_mb:.2f} MB)')

In [ ]:
# Auto-download rustren_catalog.zip to your computer
from google.colab import files
files.download('rustren_catalog.zip')